# Тема 4. Заняття 10. Практичне використання методів глибокого навчання в межах виконання індивідуальних (групових) проектів

## Навчальні питання
1. **Практичний аналіз даних в індивідуальних проектах з використанням методів глибокого навчання.**
2. **Практичний аналіз даних в групових проектах з використанням методів глибокого навчання.**

### Мета заняття
Сформувати здатність перетворювати ідею застосування deep learning на контрольований аналітичний експеримент: від постановки задачі й походження даних до baseline, архітектури, навчання, незалежного оцінювання, аналізу помилок, відтворюваності та командного review.

## 1. Місце deep learning у проекті ІАЗ

Deep learning доцільно застосовувати тоді, коли задача містить складні нелінійні залежності, великі або високорозмірні дані, зображення, послідовності чи інші структури, для яких ручне конструювання ознак є недостатнім або надто дорогим. Водночас **складніша модель не є автоматично кращою**.

Наскрізний project workflow:

`problem → data/provenance → split → baseline → DL model → training/regularization → validation → error analysis → reproducible artifact → analytical conclusion`

Ключова вимога: результат deep learning проекту має бути порівнюваним із простішою контрольною точкою та перевіреним на даних, які не використовувалися для налаштування моделі.

## 2. Project contract до написання коду

Перед створенням нейромережі зафіксуйте короткий **project contract**:

- **Аналітична проблема:** яке рішення або висновок має підтримати проект?
- **Input / target:** що надходить у модель і що вона прогнозує / класифікує / відновлює?
- **Користувач результату:** хто інтерпретує prediction і що він робить далі?
- **Дані та provenance:** джерело, версія, ліцензія/дозволеність, обмеження, synthetic/open/controlled status.
- **Baseline:** простий алгоритм або правило, з яким порівнюємо DL.
- **Метрика:** accuracy, precision/recall/F1, MAE/RMSE, AUC або task-specific metric.
- **Критерій успіху:** числовий поріг + якісні умови (стійкість, error profile, latency, explainability, human review).
- **Обмеження:** compute, час, memory, доступність GPU, data sensitivity.

Якщо ці пункти не визначені, tuning архітектури перетворюється на випадковий перебір параметрів.

## 3. Дані: split, leakage і provenance

Для supervised DL мінімально потрібні **train / validation / test**.

- `train` — навчання ваг моделі;
- `validation` — вибір архітектури, regularization, learning rate, early stopping;
- `test` — фінальна оцінка після завершення tuning.

Критичні помилки:

1. **Data leakage:** інформація з validation/test потрапляє у preprocessing або training.
2. **Near-duplicates між split-ами:** майже однакові об'єкти опиняються і в train, і в test.
3. **Temporal leakage:** майбутня інформація використовується для прогнозу минулого.
4. **Domain mismatch:** test не представляє умови майбутнього використання.
5. **Невідоме походження даних:** неможливо відтворити корпус або пояснити його обмеження.

Для image data augmentation застосовується до train і лише тоді, коли transformation не змінює semantic label.

## 4. Вибір архітектури

| Тип даних / задача | Стартова модель | Що перевіряти |
|---|---|---|
| Табличні дані | MLP + простий ML baseline | чи DL справді кращий за tree/linear model |
| Зображення | CNN / Transfer Learning | resolution, augmentation, domain shift |
| Часові послідовності | 1D-CNN / RNN/LSTM / Transformer | temporal split, horizon, drift |
| Текст | embeddings + Transformer / lightweight classifier | provenance, tokenization, class imbalance |
| Аномалії | classifier або autoencoder | false alarms, threshold, representative normal data |

Практичне правило: **почніть із найпростішої архітектури, яка може вирішити задачу**, а ускладнюйте її лише за evidence із validation/error analysis.

## 5. Навчання та regularization

Основні параметри, які слід контролювати:

- learning rate та optimizer;
- batch size;
- кількість epochs;
- width/depth моделі;
- dropout / weight decay;
- augmentation;
- early stopping;
- class weights або sampling strategy при imbalance.

Ознака overfitting: training metric продовжує покращуватися, а validation metric стабілізується або погіршується. Рішення — не «ще більше epochs», а перевірка capacity, data quality, regularization, augmentation та split validity.

In [ ]:
# Мінімальний Keras-шаблон: структура, яку адаптуємо до власного проекту
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(16, 16, 1)),
    tf.keras.layers.Conv2D(16, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(3, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 6. Evaluation: одна метрика недостатня

Фінальний review має містити щонайменше:

1. **Baseline vs DL** на однаковому test protocol.
2. Основну metric та додаткові diagnostics.
3. **Confusion matrix / error groups** для classification або residual/error distribution для regression.
4. Приклади типових помилок — не лише найкращі predictions.
5. Оцінку class imbalance / threshold trade-off.
6. Перевірку stability за seeds або повторними runs, якщо це можливо.
7. Аналіз domain shift та умов, де модель не повинна використовуватися без додаткового review.

Аналітичний висновок має відповідати не на питання «яка accuracy?», а на питання **«що модель робить надійно, де вона помиляється і чи достатньо цього для цільового сценарію?»**

# Навчальне питання 1. Індивідуальний проект

## 7. Алгоритм індивідуальної роботи

1. Сформулювати problem contract.
2. Підготувати data card: source, schema, volume, labels, limitations.
3. Створити reproducible split і зафіксувати seed.
4. Реалізувати baseline.
5. Реалізувати одну обґрунтовану DL architecture.
6. Провести 2–4 контрольовані експерименти, змінюючи **один фактор за раз**.
7. Обрати model checkpoint за validation, не test.
8. Один раз провести final test evaluation.
9. Виконати error analysis.
10. Оформити README, config, requirements, результати та короткий аналітичний brief.

### Мінімальні artifacts
- `README.md`;
- notebook/script з підготовкою даних;
- notebook/script training/evaluation;
- config або явно зафіксовані hyperparameters;
- baseline result;
- final metrics + error analysis;
- limitations + next experiment.

# Навчальне питання 2. Груповий проект

## 8. Організація групової роботи

Групова робота не повинна бути поділом notebook на випадкові частини. Команда працює з **єдиним project contract і єдиним experiment protocol**.

Рекомендовані ролі (одна людина може поєднувати кілька ролей):

- **Project / analytics lead:** problem statement, success criteria, final interpretation.
- **Data lead:** provenance, cleaning, split, data card, leakage controls.
- **Model lead:** architecture, training pipeline, checkpoints, compute.
- **Evaluation lead:** metrics, error analysis, stress tests, comparison with baseline.
- **Reproducibility / integration lead:** repository structure, configs, requirements, run instructions, merge discipline.

Критично: кожна роль залишає **перевірюваний artifact**, а не лише усне повідомлення про виконану роботу.

## 9. Group workflow і Git

Рекомендована послідовність:

1. Issue / project brief із acceptance criteria.
2. Узгоджена структура репозиторію.
3. Окремі branches для data/model/evaluation changes.
4. Невеликі commits із зрозумілими messages.
5. Pull request + peer review перед інтеграцією.
6. Єдина конфігурація experiment та versioned results table.
7. Final review: baseline → candidate → errors → limitations → recommendation.

Не зберігайте у публічному репозиторії sensitive data, credentials, access tokens, закриті model artifacts або конфіденційні результати.

## 10. Experiment table

| Run | Зміна | Train | Validation | Test | Висновок |
|---|---|---:|---:|---:|---|
| B0 | baseline | — | — | — | контрольна точка |
| D1 | базова DL model | ... | ... | *не використовувати для tuning* | чи є benefit |
| D2 | + dropout / regularization | ... | ... | — | чи зменшився gap |
| D3 | + augmentation / architecture change | ... | ... | — | чи покращилась generalization |
| FINAL | обраний checkpoint | — | — | ... | фінальний результат |

Test не є scoreboard для десятків спроб. Якщо команда багаторазово обирає рішення за test score, test фактично стає validation set.

## 11. Project review checklist

Перед захистом перевірте:

- [ ] проблема і target user сформульовані;
- [ ] provenance та обмеження даних описані;
- [ ] train/validation/test розділені коректно;
- [ ] є baseline;
- [ ] architecture вибрана за task/data, а не за популярністю;
- [ ] tuning не використовує test;
- [ ] є regularization / early stopping strategy;
- [ ] показано не лише aggregate metric, а й errors;
- [ ] documented seeds/config/environment;
- [ ] є limitations і next step;
- [ ] для групи видно внесок ролей і review history.

## 12. Питання для самоконтролю

1. Чому baseline потрібен навіть у deep learning проекті?
2. Чим validation відрізняється від final test?
3. Які ознаки data leakage можливі у вашому типі даних?
4. Як визначити, що model capacity завелика?
5. Чому accuracy може бути недостатньою для imbalanced classification?
6. Які artifacts роблять індивідуальний проект відтворюваним?
7. Які artifacts має залишити кожна роль у груповому проекті?
8. За якою evidence можна обґрунтувати, що DL справді потрібне?

### Підсумок
Успішний deep learning проект — це не найбільша модель і не найвище training score. Це **перевірюваний ланцюг evidence**, у якому задача, дані, baseline, protocol, модель, помилки, обмеження та висновок узгоджені між собою.